# 5. State Callback (Expanded)

Receives robot state and captures initial joint positions.

---

```python
def LowStateHandler(self, msg):
    self.low_state = msg

    if not self.first_update:
        self.first_update = True
        self.initial_pose = [msg.motor_state[j].q for j in self.joints]
```

---

## 🧠 Big Picture: What Is This Function?

This function is a **DDS callback** that is automatically triggered whenever the robot publishes new state data.

```text
Robot → DDS → calls this function → gives you state
```

---

## 🔁 When Does This Run?

Every time the robot sends a `LowState` message:

```text
Typically at ~500 Hz (internal)
Delivered to you at ~50–100 Hz (effective)
```

So this function is called:

> 🔄 **continuously and asynchronously**

---

## 📥 Input: `msg`

### What is `msg`?

```python
msg : LowState_
```

This is a structured message containing:

* Joint positions (`q`)
* Joint velocities (`dq`)
* Motor states
* System status

---

### Example access:

```python
msg.motor_state[15].q
```

→ Position of joint 15 (LeftShoulderPitch)

---

## 🧱 Step 1: Store the Full State

```python
self.low_state = msg
```

---

### Why store it?

This makes the latest robot state available to the rest of your program:

```python
self.low_state.motor_state[j].q
```

---

### 🧠 Important Insight

This line turns your program into a **stateful system**:

```text
Before: no knowledge of robot  
After: continuously updated state model  
```

---

## ⚠️ Asynchronous Execution (Critical Concept)

This function runs in a **different thread** than your control loop.

```text
Thread 1 → ControlLoop (your commands)
Thread 2 → LowStateHandler (incoming data)
```

---

### Implication:

```text
ControlLoop reads state while it is being updated
```

In practice:

* Usually safe
* But conceptually important for advanced systems

---

## 🚦 Step 2: First Update Check

```python
if not self.first_update:
```

---

### Why is this needed?

At startup:

```text
self.low_state = None
```

If you try to use it:

> ❌ You crash or send invalid commands

---

### So we wait until:

```text
We have at least ONE valid state message
```

---

## ✅ Step 3: Capture Initial Pose

```python
self.initial_pose = [msg.motor_state[j].q for j in self.joints]
```

---

### What does this do?

It reads the current joint angles:

```text
For each joint j:
    get current position q
```

---

### Example result:

```python
[-0.02, 0.15, 0.01, -0.8, 0.0, 0.3, 0.0]
```

---

## 🧠 Why This Is CRITICAL

This ensures:

> 🎯 Your motion starts from the **robot’s actual physical pose**

---

### Without this:

If you assumed:

```python
initial_pose = [0, 0, 0, 0, 0, 0, 0]
```

But the robot is actually:

```python
[0.2, -0.1, ...]
```

→ The robot would:

* Jump suddenly
* Generate large torques
* Potentially destabilize

---

## 🔬 Engineering Insight

This is a form of:

> **State synchronization**

You are aligning:

```text
Controller state ↔ Physical robot state
```

---

## 🧠 Why Only Do This Once?

```python
self.first_update = True
```

---

### After first capture:

* You already know the starting pose
* No need to overwrite it again

---

### If you DID overwrite continuously:

```text
initial_pose would keep changing
```

→ Your trajectory would:

* Drift
* Become unstable

---

## 🔄 Relationship to Control Loop

Later in your program:

```python
self.interp(self.initial_pose[i], self.target_raise[i], r)
```

---

### This means:

```text
Motion = interpolate(current_pose → target_pose
```

So this line defines:

> 🎯 The **starting point of all motion**

---

## 🤖 RL Interpretation

This function defines your:

```text
state = msg.motor_state
```

---

### In RL terms:

```python
state = [joint positions]
```

This is what your policy will eventually use:

```python
action = policy(state)
```

---

## ⚠️ Subtle but Important Detail

You are selecting only:

```python
for j in self.joints
```

---

### Meaning:

You are extracting:

> Only the **left arm state**

Not:

* legs
* torso
* right arm

---

### This defines your:

```text
Observation space (subset of full robot state)
```

---

## 🔥 Hidden System Behavior

Even though you only use arm joints:

* The robot still publishes **full-body state**
* You are choosing a **projection of that state**

---

## 🚀 Summary

This function:

| Step                   | Purpose                      |
| ---------------------- | ---------------------------- |
| Receive `msg`          | Get robot telemetry          |
| Store `low_state`      | Make state accessible        |
| Check first update     | Ensure safe startup          |
| Capture `initial_pose` | Define motion starting point |

---

## 🧠 Teaching Insight

This is where you can emphasize:

> “Control starts with perception.”

Before you move the robot, you must:

* Observe it
* Understand its current state
* Synchronize with reality

---

## 🔗 Connection to Entire System

This function enables:

```text
LowState → ControlLoop → LowCmd
```

Without it:

> ❌ No feedback
> ❌ No safe motion
> ❌ No RL

